# S1.3 — Docker Infrastructure

Validates the Docker Compose setup, Dockerfile, and Makefile for PaperAlchemy.

**Services**: PostgreSQL, Redis, OpenSearch, Ollama, Airflow, Langfuse stack (6 services), pgAdmin, OpenSearch Dashboards

**Profiles**: `(default)` = core, `full` = api+ollama+airflow, `langfuse` = observability, `dev-tools` = dashboards

In [1]:
import yaml
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent
compose = yaml.safe_load((PROJECT_ROOT / 'compose.yml').read_text())
print(f"Services defined: {len(compose['services'])}")
for name in sorted(compose['services'].keys()):
    profiles = compose['services'][name].get('profiles', ['(default)'])
    has_hc = 'healthcheck' in compose['services'][name]
    print(f"  {name:30s} profiles={profiles!s:30s} healthcheck={'yes' if has_hc else 'NO'}")

Services defined: 14
  airflow                        profiles=['full']                       healthcheck=yes
  api                            profiles=['full']                       healthcheck=yes
  clickhouse                     profiles=['langfuse']                   healthcheck=yes
  langfuse-minio                 profiles=['langfuse']                   healthcheck=yes
  langfuse-postgres              profiles=['langfuse']                   healthcheck=yes
  langfuse-redis                 profiles=['langfuse']                   healthcheck=yes
  langfuse-web                   profiles=['langfuse']                   healthcheck=yes
  langfuse-worker                profiles=['langfuse']                   healthcheck=yes
  ollama                         profiles=['full']                       healthcheck=yes
  opensearch                     profiles=['(default)']                  healthcheck=yes
  opensearch-dashboards          profiles=['dev-tools']                  healthcheck=yes


In [2]:
# Verify profile groupings
profiles = {}
for name, config in compose['services'].items():
    for p in config.get('profiles', ['(default)']):
        profiles.setdefault(p, []).append(name)

for profile, services in sorted(profiles.items()):
    print(f"\nProfile: {profile}")
    for s in services:
        print(f"  - {s}")


Profile: (default)
  - postgres
  - redis
  - opensearch

Profile: dev-tools
  - opensearch-dashboards
  - pgadmin

Profile: full
  - api
  - ollama
  - airflow

Profile: langfuse
  - clickhouse
  - langfuse-postgres
  - langfuse-redis
  - langfuse-minio
  - langfuse-web
  - langfuse-worker


In [3]:
# Verify volumes
volumes = list(compose.get('volumes', {}).keys())
print(f"Named volumes ({len(volumes)}):")
for v in volumes:
    print(f"  - {v}")

Named volumes (8):
  - postgres_data
  - redis_data
  - opensearch_data
  - ollama_data
  - airflow_logs
  - clickhouse_data
  - langfuse_postgres_data
  - langfuse_minio_data


In [4]:
# Verify Dockerfile stages
dockerfile = (PROJECT_ROOT / 'Dockerfile').read_text()
stages = [line for line in dockerfile.split('\n') if line.strip().startswith('FROM')]
print("Dockerfile stages:")
for s in stages:
    print(f"  {s.strip()}")

Dockerfile stages:
  FROM ghcr.io/astral-sh/uv:python3.12-bookworm AS base
  FROM python:3.12.8-slim AS final


In [5]:
# Verify Makefile targets
makefile = (PROJECT_ROOT / 'Makefile').read_text()
targets = [line.split(':')[0] for line in makefile.split('\n') if ':' in line and not line.startswith('\t') and not line.startswith('#') and not line.startswith('.')]
print(f"Makefile targets ({len(targets)}):")
for t in sorted(set(targets)):
    if t.strip():
        print(f"  make {t.strip()}")

Makefile targets (20):
  make build
  make clean
  make down
  make down-clean
  make format
  make gradio
  make health
  make help
  make lint
  make logs
  make restart
  make setup
  make start
  make status
  make stop
  make test
  make test-cov
  make up
  make up-all
  make up-langfuse


In [6]:
# Quick Docker validation (requires Docker)
import subprocess
result = subprocess.run(
    ['docker', 'compose', 'config', '--quiet'],
    cwd=PROJECT_ROOT,
    capture_output=True, text=True
)
if result.returncode == 0:
    print('docker compose config: VALID')
else:
    print(f'docker compose config: FAILED\n{result.stderr}')

docker compose config: VALID
